<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - PWN, or Binary Exploitation</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 4: ROP
</h1>

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">1 - Second defense: NX</h1>

Well, stack cookies weren't enough to stop hackers. Leaking them out still allows you to shellcode to your heart's content and that's no good (for the folks working against us). Enter the next level of binary security: non-executable regions of memory.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.1 - Memory regions refresher</h2>

Remember memory regions from shellcoding? No? Go back and read that section, I put a lot of work into writing it. Just kidding, here's the important bit: sections of bytes within the binary are loaded at different ranges of memory addresses, each with different Read/Write/eXecute permissions. In the shellcoding section, we assumed that the stack will always have executable permissions. Now, we assume the opposite!

In [2]:
!mkdir -p ~/introsec/pwn/rop/binaries

In [3]:
%cd ~/introsec/pwn/rop/binaries

/home/user/introsec/pwn/rop/binaries


Let's take a look at a simple C program using the tool `checksec`:

In [4]:
%%file example.c

#include <stdio.h>
#include <stdlib.h>

int main(void){
    printf("Hello, world!");
    return 0;
}

Overwriting example.c


In [5]:
!gcc example.c -o example -no-pie

In [6]:
!checksec --file=example

[*] '/home/user/introsec/pwn/rop/binaries/example'
    Arch:       amd64-64-little
    RELRO:      Partial RELRO
    Stack:      No canary found
    NX:         NX enabled
    PIE:        No PIE (0x400000)
    SHSTK:      Enabled
    IBT:        Enabled
    Stripped:   No


`checksec` will list a number of protections that are or are not found within a binary. Notice the indicators for RELRO, Stack canary (cookie), and PIE. However, the one we care about right now is `NX`. What this does is removes the `x` permission from the stack region of memory. If you look at the above output from `vmmap`, the stack only has `rw` permissions.

(I am aware that the images below list the binaries in the shellcoding directory :P. They will be created in the rop directory on your machine.)

If we look at the result of `vmmap` for `example`:

<img src="./media/example_vmmap.png">

We see again that the purple `stack` section of memory does not have executable permissions. Let's try recompiling with `NX` disabled and examining with `checksec` and `GDB`

In [7]:
!gcc example.c -z execstack -o vulnerable_example -no-pie

In [8]:
!checksec --file=vulnerable_example

[*] '/home/user/introsec/pwn/rop/binaries/vulnerable_example'
    Arch:       amd64-64-little
    RELRO:      Partial RELRO
    Stack:      No canary found
    NX:         NX unknown - GNU_STACK missing
    PIE:        No PIE (0x400000)
    Stack:      Executable
    RWX:        Has RWX segments
    SHSTK:      Enabled
    IBT:        Enabled
    Stripped:   No


<img src="./media/vulnerable_example_vmmap.png">

As we now see, `checksec` displays that the stack is executable, `rwx` segments exist, and `NX` is unidentifiable.

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - Second strike: ROP</h1>

Of course, if simply disabling shellcode via memory permissions stopped all the hackers in the world then we wouldn't be here teaching you about how to counter it. So, what is ROP?

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.1 - How to program in a return-oriented fashion</h2>

The idea is simple: We can't write our own code to a region of memory and execute it, but we can write the addresses of code that already exists in the binary and jump to that!

This may not be exceedingly clear at first, so here is what I mean: the actual bytes representing the assembly are stored in the regions of memory marked with the `X` permission such that the can be executed. Take a look at the below example assembly:

```c
mov rax, 0x5
add rax, rbx
sal rax, 2
```

The first instruction in that list (`mov rax, 0x5`) is stored at `0x401000` (in this example). Let's say that the region `0x400000` to `0x40a000` contains legitimate code that is executable. In previous exploits, we would overwrite the return address of a function with the address of our shellcode. This is no longer possible, but we can certainly call code within the range mentioned above. For example, if we overwrite the return address of a function with `0x401000`, we can execute the above three instructions!

Now, are those three instructions actually useful? No, not in this context. However, there is one that you will come to love: `pop`!

Recall the goal of shellcoding: to initialize `RDI`, `RSI`, `RDX`, and `RAX` to the address of the string `/bin/sh`, 0, 0, and `0x3b` respectively before calling `syscall`. Say hello to gadgets!

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.2 - (Inspector) Gadgets </h2>

`Gadgets` are the colloquial term used to describe instructions ending in `ret` (or in some cases, `call`/`jmp`/`syscall`). We'll talk about why they end like that specifically in just a moment, but for now, just take it as an axiom.

There isn't much more to say here, I just wanted to introduce this bit of terminology :).

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.3 - Chains! Clink clank</h2>

Now we can clarify why gadgets end the way they do. See, shellcode is easy to follow: you write it to the stack, you jump to the start, and it executes in a nice, clean, linear fashion. With ROP, this is not the case. Remember, you only get control of `RIP` once when you overwrite the return address but you need to execute a few instructions (usually) to set up your registers and `syscall`. How can we do this?

Let's say the following gadgets exist at the following addresses:

```c
0x401000:
pop rdi
ret
```

```c
0x402000:
pop rsi
ret
```

These are very useful gadgets! See, we may not be able to execute code on the stack, but we can still (ideally) write whatever we want there! Following this, we can use `pop`s as our main method of moving data where we want. Take a look at the below example of a stack:

```c
0x7ffffff018: 0x9abcdef0
0x7ffffff010: 0x402000
0x7ffffff008: 0x12345678
0x7ffffff000: 0x401000
```

Let's assume that this is the stack right as you get to the `ret` of some vulnerable function. Recall that `ret` is an alias for `pop rip`. After our vulnerable function returns, our stack will become

```c
0x7ffffff018: 0x9abcdef0
0x7ffffff010: 0x402000
0x7ffffff008: 0x12345678
```

and our current instructions are

```c
0x401000:
pop rdi
ret
```

We will hit our `pop rdi`, setting it to `0x12345678` (i.e. some address containing `/bin/sh`). The stack becomes

```c
0x7ffffff018: 0x9abcdef0
0x7ffffff010: 0x402000
```

followed by a `ret` which makes the stack

```c
0x7ffffff018: 0x9abcdef0
```

and our current instructions

```c
0x402000:
pop rsi
ret
```

A string or "chain" of gadgets like this is called a `ROP Chain`! Does the idea make sense yet? If not, don't worry! Ask us questions, re-read the section, but most importantly, try it out! This is a very complicated exploit technique that most often only makes sense after you've practiced and succeeded in implementing it a few times.

Ask questions!!!

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.4 - Where are the gadgets?</h2>

There are two tools I can recommend for finding ROP gadgets. These are `ropper` and `ROPgadget`. The long made short of what to use is as follows:

- `ROPgadget` is easier to use, lending itself to newcomers (probably what you want to start with)
- `ropper` will find gadgets that `ROPgadget` will miss (i.e. `ROPgadget` will find fewer gadgets), but is a little more complex of a use case (though this allows for more fine-tuned control over your searches)

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">3 - Advanced ROP</h1>

Let's talk about a particular ROP technique that you are likely to find useful: the `stack pivot`.

If you'll recall from stack frame cleanup, there is another important instruction that (usually) executes before `ret`, that being `leave`. As a reminder, `leave` is aliased to:

```c
mov rsp, rbp
pop rbp
```

What this means for us is that the stack may not be pointing where we want it to anymore. If `RBP` contains an address that does not align properly with our chain then nothing will happen after `ret` occurs. The way we get around this is simple: control `RBP` (and also know where our data is on the stack).

By controlling `RBP` before `leave` occurs, we can control where the stack will point to before the function returns. This allows us to position the stack to be aligned with our chain such that when `ret` occurs (remember, `pop rip`), our first gadget will be executed and our data will follow.

Simple in theory, messy and sometimes hard to understand/debug in practice. Ask questions!

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Putting it all together</h1>

There are 5 challenges in `./binaries/chals.` The first one, `rop00`, is a warm-up BOF. Afterwards, you can slowly work your way up the ROP chain (pun intended) of techniques. I recommend completing all of them to get comfortable with the process!!